In [1]:
# Thiết lập thư mục gốc một lần để notebook chạy được cả trên máy cá nhân và Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [2]:
# Nạp các mô-đun trong repository và dùng các thư mục kết quả đã lưu.
import glob, hashlib, json, os, sys
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from src.models.coldrec_adapter import ColdRecDataAdapter

SCORES = str(DATA_DIR / "scores")
OUT = str(DATA_DIR / "results")
os.makedirs(OUT, exist_ok=True)


In [3]:
# Cấu hình ánh xạ các thư mục mô hình + hàm đọc/kiểm tra tệp điểm
N = {None: 3340, "trung": 1028, "moi": 2844}
CORE = ["knn", "dropoutnet", "aldi", "semproj"]  # mô hình bắt buộc hiện có
EXTRA = []  # USIM và CGRC không thuộc các bảng kết quả
TRAINED = ["dropoutnet", "aldi", "semproj"]
E3_MODELS = ["dropoutnet", "aldi", "semproj"]
ROB = "_robust3s"  # chọn cấu hình bằng 3 seed

MODEL_SUBDIRS = {
    "knn": "Content-KNN",
    "dropoutnet": "DropoutNet",
    "aldi": "ALDI",
    "semproj": "SemProj",
    "usim": "USIM",
    "cgrc": "CGRC",
}


def path(m, inp, g=None, tag=""):
  fname = f"user_ndcg20_{m}_{inp}{tag}" + (f"_{g}" if g else "") + ".npy"
  subdir = MODEL_SUBDIRS.get(m, m)

  # 1. Tìm đúng tên tệp trong thư mục con hoặc thư mục gốc
  candidates = [os.path.join(SCORES, subdir, fname), os.path.join(SCORES, fname)]
  if os.path.exists(SCORES):
    for d in os.listdir(SCORES):
      full_d = os.path.join(SCORES, d)
      if os.path.isdir(full_d) and d.lower() in (m.lower(), subdir.lower()):
        candidates.append(os.path.join(full_d, fname))
  for c in candidates:
    if os.path.exists(c):
      return c

  # 2. Với ALDI, khi chọn bằng 3 seed thì raw và fused
  #    giữ nguyên cấu hình như 1 seed -> fallback về tệp 1 seed nếu không lưu bản sao riêng
  if m == "aldi" and tag == ROB and inp in ("raw", "fused"):
    return path(m, inp, g=g, tag="")

  return candidates[0]


def has_all(m, inps, groups=(None,), tag=""):
  return all(os.path.exists(path(m, i, g, tag)) for i in inps for g in groups)


def load(m, inp, g=None, tag=""):
  p = path(m, inp, g, tag)
  assert os.path.exists(p), f"THIẾU TỆP: {p}"
  a = np.load(p).astype(np.float64)
  assert a.shape == (N[g],), f"{p}: kích thước {a.shape}, cần {(N[g],)}"
  assert np.all((a >= 0) & (a <= 1 + 1e-9)), f"{p}: NDCG ngoài [0, 1]"
  return a


# Liệt kê nhanh trạng thái các thư mục con trong SCORES
for m in CORE + EXTRA:
  sdir = os.path.join(SCORES, MODEL_SUBDIRS[m])
  n_files = len(glob.glob(f"{sdir}/*.npy")) if os.path.exists(sdir) else 0
  print(f"  - {MODEL_SUBDIRS[m]:<12} ({m:<10}): {n_files:2d} tệp .npy")

  - Content-KNN  (knn       ):  9 tệp .npy
  - DropoutNet   (dropoutnet): 18 tệp .npy
  - ALDI         (aldi      ): 18 tệp .npy
  - SemProj      (semproj   ): 14 tệp .npy


In [4]:
# Thực thi kiểm định thống kê hợp nhất & xuất bảng 
from src.models.coldrec_adapter import ColdRecDataAdapter

# 1) Tệp điểm phải khớp bảng kết quả (trung bình theo học viên = trung bình 5 seed)
EXPECTED = {
    ("knn", "raw"): 0.1053,
    ("knn", "llm"): 0.0819,
    ("knn", "fused"): 0.1054,
    ("dropoutnet", "raw"): 0.0827,
    ("dropoutnet", "llm"): 0.0737,
    ("dropoutnet", "fused"): 0.0930,
    ("dropoutnet", "llm_3d"): 0.0751,
    ("dropoutnet", "fused_3d"): 0.0886,
    ("aldi", "raw"): 0.0834,
    ("aldi", "llm"): 0.0746,
    ("aldi", "fused"): 0.0815,
    ("aldi", "llm_3d"): 0.0748,
    ("aldi", "fused_3d"): 0.0806,
    ("aldi", "llm_robust3s"): 0.0878,
    ("semproj", "raw"): 0.0750,
    ("semproj", "llm"): 0.0978,
    ("semproj", "fused"): 0.0892,
    ("semproj", "llm_3d"): 0.0860,
    ("semproj", "fused_3d"): 0.0770,
}
for (m, inp), v in EXPECTED.items():
  got = load(m, inp).mean()
  assert abs(got - v) < 1e-4, f"{m}/{inp}: tệp cho {got:.4f}, bảng ghi {v:.4f}"
print(
    "[OK] Toàn bộ tệp điểm trong EXPECTED đều khớp bảng kết quả (sai lệch <"
    " 1e-4)."
)


# 2) Kiểm định: Wilcoxon hai phía, Holm trong họ, CI bootstrap 95%, quy tắc W9
def compare(a, b, n_boot=10000, seed=0):
  d = a - b
  p = wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue
  rng = np.random.default_rng(seed)
  bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
  lo, hi = np.percentile(bs, [2.5, 97.5])
  return dict(
      diff=d.mean(),
      rel=d.mean() / b.mean(),
      lo=lo,
      hi=hi,
      n_khac_0=int((d != 0).sum()),
      p=p,
  )


def holm(p):
  p = np.asarray(p, float)
  m = len(p)
  out = np.empty(m)
  run = 0.0
  for r, i in enumerate(np.argsort(p)):
    run = max(run, (m - r) * p[i])
    out[i] = min(1.0, run)
  return out


def label(r):
  # Giữ nguyên chữ thường để khớp hoàn toàn với các hàm verdict
  sig, ci = r["p_holm"] < 0.05, (r["lo"] > 0) or (r["hi"] < 0)
  if sig and ci:
    return "tăng có ý nghĩa" if r["diff"] > 0 else "giảm có ý nghĩa"
  return "không khác biệt" if not (sig or ci) else "không kết luận được"


def family(name, pairs):
  rows = [dict(ho=name, so_sanh=k, **compare(a, b)) for k, (a, b) in pairs.items()]
  for r, pa in zip(rows, holm([r["p"] for r in rows])):
    r["p_holm"] = pa
    r["ket_luan"] = label(r)
  return rows


# 3) E-RQ1: họ chính (gốc + LLM so với gốc) và họ phụ (chỉ LLM so với gốc)
M1 = CORE + [m for m in EXTRA if has_all(m, ["raw", "llm", "fused"])]
e1 = family("E1_chinh", {m: (load(m, "fused"), load(m, "raw")) for m in M1})
e1p = family("E1_phu", {m: (load(m, "llm"), load(m, "raw")) for m in M1})


def robust(m, inp):
  if not has_all(m, [inp, "raw"], tag=ROB):
    return None
  r = compare(load(m, inp, tag=ROB), load(m, "raw", tag=ROB))
  main = {x["so_sanh"]: x for x in (e1 if inp == "fused" else e1p)}[m]
  return dict(giu_chieu=bool(np.sign(r["diff"]) == np.sign(main["diff"])), **r)


def verdict_e1():
  r = {x["so_sanh"]: x for x in e1}
  sig = {m for m in r if r[m]["ket_luan"] == "tăng có ý nghĩa"}
  rb = {
      m: True if m == "knn" else (robust(m, "fused") or {}).get("giu_chieu")
      for m in sig
  }
  pending = sorted(m for m, v in rb.items() if v is None)
  up = {m for m, v in rb.items() if v}
  trained = {m for m in r if m in TRAINED}

  def level(u):
    if trained <= u and r["knn"]["ket_luan"] != "giảm có ý nghĩa":
      return "Cải thiện nhất quán"
    return "Phụ thuộc kiến trúc" if u else "Không có bằng chứng"

  v = level(up) if not pending else f"CHƯA CHỐT: thiếu độ bền {pending}"
  return dict(
      ket_luan=v,
      neu_du_do_ben=level(up | set(pending)),
      cai_thien=sorted(up),
      mo_hinh=sorted(r),
  )


# 4) E-RQ2: Holm theo nhóm. CI của (mới - trùng): bootstrap theo học viên trên toàn 3.340 HV
ad = ColdRecDataAdapter()
users = sorted(ad.test_target_dict)
in_g = {
    g: np.array([any(i in s for i in ad.test_target_dict[u]) for u in users])
    for g, s in ad.groups_prq3.items()
}
assert in_g["trung"].sum() == 1028 and in_g["moi"].sum() == 2844

M2 = CORE + [
    m for m in EXTRA if has_all(m, ["raw", "fused"], ("moi", "trung"))
]
e2 = {
    g: family(
        f"E2_{g}", {m: (load(m, "fused", g), load(m, "raw", g)) for m in M2}
    )
    for g in ("moi", "trung")
}


def dd_ci(m, n_boot=10000, seed=0):
  d = {}
  for g in ("moi", "trung"):
    d[g] = np.full(len(users), np.nan)
    d[g][in_g[g]] = load(m, "fused", g) - load(m, "raw", g)
  rng = np.random.default_rng(seed)
  bs = np.empty(n_boot)
  for b in range(n_boot):
    i = rng.integers(0, len(users), len(users))
    bs[b] = np.nanmean(d["moi"][i]) - np.nanmean(d["trung"][i])
  lo, hi = np.percentile(bs, [2.5, 97.5])
  return dict(diff=np.nanmean(d["moi"]) - np.nanmean(d["trung"]), lo=lo, hi=hi)


dd = [dict(ho="E2_moi-trung", so_sanh=m, **dd_ci(m)) for m in M2]


def verdict_e2():
  moi = {x["so_sanh"]: x for x in e2["moi"]}
  ok = {
      r["so_sanh"]: bool(
          moi[r["so_sanh"]]["ket_luan"] == "tăng có ý nghĩa" and r["lo"] > 0
      )
      for r in dd
      if r["so_sanh"] in TRAINED
  }
  v = (
      "Tập trung ở khóa mới"
      if sum(ok.values()) > len(ok) / 2
      else "Mô tả từng mô hình"
  )
  return dict(ket_luan=v, dat=ok)


# 5) E-RQ3: MỘT họ Holm gồm 6 so sánh (3 mô hình x 2 đầu vào)
e3 = family(
    "E3",
    {
        f"{m}|{inp}": (load(m, inp), load(m, f"{inp}_3d"))
        for m in E3_MODELS
        for inp in ("fused", "llm")
    },
)


def verdict_e3():
  kl = {}
  for r in e3:
    kl.setdefault(r["so_sanh"].split("|")[0], []).append(r["ket_luan"])
  up = [m for m, v in kl.items() if "tăng có ý nghĩa" in v]
  down = [m for m, v in kl.items() if "giảm có ý nghĩa" in v]
  v = "Có đóng góp" if len(up) >= 2 and not down else "Chưa có bằng chứng đóng góp"
  return dict(ket_luan=v, tang=up, giam=down)


# 6) Bảng độ bền cho mọi so sánh có thể được nêu tên trong bài
rob = []
for m in [x for x in M1 if x in TRAINED]:
  for inp in ("fused", "llm"):
    r = robust(m, inp)
    rob.append(
        dict(ho="Do_ben", so_sanh=f"{m}|{inp}-raw", **(r or {"diff": np.nan}))
    )

# 7) Lưu tệp results/final_stats.csv, verdicts.json và manifest_scores.json
rows = e1 + e1p + e2["moi"] + e2["trung"] + dd + e3 + rob
df_raw = pd.DataFrame(rows)
df_raw.to_csv(f"{OUT}/final_stats.csv", index=False, encoding="utf-8-sig")

V = {"E-RQ1": verdict_e1(), "E-RQ2": verdict_e2(), "E-RQ3": verdict_e3()}
json.dump(
    V,
    open(f"{OUT}/verdicts.json", "w", encoding="utf-8"),
    ensure_ascii=False,
    indent=1,
)

all_npy = sorted(glob.glob(f"{SCORES}/**/*.npy", recursive=True))
md5 = {
    os.path.basename(p): hashlib.md5(open(p, "rb").read()).hexdigest()
    for p in all_npy
}
json.dump(
    md5, open(f"{OUT}/manifest_scores.json", "w", encoding="utf-8"), indent=1
)


# 8) ĐỊNH DẠNG BẢNG TRÌNH BÀY & CHÚ THÍCH 
MODEL_NAME_MAP = {
    "knn": "Content-KNN",
    "dropoutnet": "DropoutNet",
    "aldi": "ALDI",
    "semproj": "SemProj",
    "usim": "USIM",
    "cgrc": "CGRC",
}


def fmt_num(x, decimals=4, signed=True):
  if pd.isna(x):
    return "—"
  s = f"{x:+.{decimals}f}" if signed else f"{x:.{decimals}f}"
  return s.replace("-", "−").replace(".", ",")


def fmt_pct(rel):
  if pd.isna(rel):
    return "—"
  pct = rel * 100.0
  if abs(pct) < 0.05:
    return "≈ 0%"
  return f"{pct:+.1f}%".replace("-", "−").replace(".", ",")


def fmt_p(p):
  if pd.isna(p):
    return "—"
  if p >= 0.01:
    return f"{p:.2f}".replace(".", ",")
  exp = int(np.floor(np.log10(abs(p))))
  mant = p / (10**exp)
  sup = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")
  return f"{mant:.1f} × 10{str(exp).translate(sup)}".replace(".", ",")


def fmt_int_vn(n):
  if pd.isna(n):
    return "—"
  return f"{int(n):,}".replace(",", ".")


def fmt_cell_stat(r, show_hv=True):
  if pd.isna(r.get("diff")):
    return "—"
  star = (
      "*"
      if r.get("ket_luan") in ("tăng có ý nghĩa", "giảm có ý nghĩa")
      else ""
  )
  base = (
      f"{fmt_num(r['diff'])}{star} ({fmt_pct(r.get('rel'))})"
      f" [{fmt_num(r['lo'])}; {fmt_num(r['hi'])}]"
  )
  if pd.isna(r.get("p_holm")):
    if not pd.isna(r.get("p")):
      return (
          f"{base}; p_Wilcoxon = {fmt_p(r['p'])};"
          f" {fmt_int_vn(r.get('n_khac_0'))} HV khác 0"
      )
    return f"{fmt_num(r['diff'])} [{fmt_num(r['lo'])}; {fmt_num(r['hi'])}]"
  hv_str = (
      f"; {fmt_int_vn(r.get('n_khac_0'))} HV khác 0"
      if show_hv and not pd.isna(r.get("n_khac_0"))
      else ""
  )
  return f"{base}; p_Holm = {fmt_p(r['p_holm'])}{hv_str}"


# Bảng chuẩn hóa trực tiếp từ final_stats.csv kèm chú thích đầy đủ ở dòng cuối
HO_PRETTY = {
    "E1_chinh": "E-RQ1 (Họ chính: Gốc + LLM so với Gốc)",
    "E1_phu": "E-RQ1 (Họ phụ: Chỉ LLM so với Gốc)",
    "E2_moi": "E-RQ2 (Nhóm khóa mới: Gốc + LLM so với Gốc)",
    "E2_trung": "E-RQ2 (Nhóm khóa trùng: Gốc + LLM so với Gốc)",
    "E2_moi-trung": "E-RQ2 (Hiệu số: Nhóm mới − Nhóm trùng)",
    "E3": "E-RQ3 (Hồ sơ 5 chiều so với 3 chiều)",
    "Do_ben": "Phân tích độ bền (Chọn cấu hình bằng 3 seeds)",
}


def pretty_comp(val):
  parts = str(val).split("|")
  m_name = MODEL_NAME_MAP.get(parts[0], parts[0])
  if len(parts) == 1:
    return m_name
  sfx = {
      "fused": "Gốc + LLM: 5 chiều − 3 chiều",
      "llm": "Chỉ LLM: 5 chiều − 3 chiều",
      "fused-raw": "Gốc + LLM so với Gốc (3 seeds)",
      "llm-raw": "Chỉ LLM so với Gốc (3 seeds)",
  }.get(parts[1], parts[1])
  return f"{m_name} ({sfx})"


df_pres = pd.DataFrame()
df_pres["Họ kiểm định"] = df_raw["ho"].map(HO_PRETTY)
df_pres["Mô hình / So sánh"] = df_raw["so_sanh"].apply(pretty_comp)
df_pres["Δ NDCG@20 (Mean)"] = df_raw["diff"].apply(lambda x: fmt_num(x, 4, True))
df_pres["Chênh lệch tương đối (%)"] = df_raw["rel"].apply(fmt_pct)
df_pres["CI Bootstrap 95%"] = [
    f"[{fmt_num(l)}; {fmt_num(h)}]" if not pd.isna(l) else "—"
    for l, h in zip(df_raw["lo"], df_raw["hi"])
]
df_pres["Số HV khác 0"] = df_raw["n_khac_0"].apply(fmt_int_vn)
df_pres["p (Wilcoxon)"] = df_raw["p"].apply(fmt_p)
df_pres["p_Holm (Theo họ)"] = df_raw["p_holm"].apply(fmt_p)
df_pres["Trình bày chuẩn"] = [
    fmt_cell_stat(r) for _, r in df_raw.iterrows()
]
df_pres["Kết luận"] = (
    df_raw["ket_luan"].fillna("—").str.capitalize()
)
df_pres["Giữ chiều (Độ bền 3 seeds)"] = df_raw["giu_chieu"].map(
    {True: "Giữ chiều (True)", False: "Đổi chiều (False)"}
).fillna("—")

# Dòng chú thích 
NOTE_MUC_5_4 = (
    "Chú thích: Số liệu tính trên trung bình 5 seeds"
    " huấn luyện (42, 2023, 2024, 1007, 999) với độ lệch chuẩn mẫu mean ± std"
    " (ddof = 1); Content-KNN tất định. Kiểm định Wilcoxon bắt cặp hai phía"
    " theo từng học viên (n = 3.340 học viên toàn tập; n = 2.844 học viên nhóm"
    " khóa mới; n = 1.028 học viên nhóm khóa trùng; 532 học viên thuộc cả hai"
    " nhóm). Giá trị p được hiệu chỉnh Holm theo từng họ kiểm định (E-RQ1 họ"
    " chính: 4 so sánh; E-RQ1 họ phụ: 4 so sánh; E-RQ2 nhóm mới: 4 so sánh;"
    " E-RQ2 nhóm trùng: 4 so sánh; E-RQ3: 1 họ 6 so sánh). Khoảng tin cậy CI"
    " bootstrap 95% tính từ 10.000 lần lấy mẫu lại theo học viên (seed = 0)."
    " Dấu * biểu thị khác biệt có ý nghĩa thống kê (đồng thời"
    " p_Holm < 0,05 và CI 95% không chứa 0)."
)

note_row = {col: "" for col in df_pres.columns}
note_row["Họ kiểm định"] = NOTE_MUC_5_4
df_pres_with_note = pd.concat(
    [df_pres, pd.DataFrame([note_row])], ignore_index=True
)
df_pres_with_note.to_csv(
    f"{OUT}/final_stats_tables_with_notes.csv",
    index=False,
    encoding="utf-8-sig",
)

print("\n=== BẢNG KIỂM ĐỊNH TRÌNH BÀY ===")
print(
    df_pres[[
        "Họ kiểm định",
        "Mô hình / So sánh",
        "Trình bày chuẩn",
        "Kết luận",
        "Giữ chiều (Độ bền 3 seeds)",
    ]].to_string(index=False)
)
print(f"\n{NOTE_MUC_5_4}")

print("\n=== VERDICTS.JSON ===")
print(json.dumps(V, ensure_ascii=False, indent=1))

[OK] Toàn bộ tệp điểm trong EXPECTED đều khớp bảng kết quả (sai lệch < 1e-4).


=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===


MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c


=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===




=== BẢNG KIỂM ĐỊNH TRÌNH BÀY ===
                                 Họ kiểm định                           Mô hình / So sánh                                                                Trình bày chuẩn            Kết luận Giữ chiều (Độ bền 3 seeds)
       E-RQ1 (Họ chính: Gốc + LLM so với Gốc)                                 Content-KNN               +0,0001 (+0,1%) [−0,0033; +0,0034]; p_Holm = 1,00; 639 HV khác 0     Không khác biệt                          —
       E-RQ1 (Họ chính: Gốc + LLM so với Gốc)                                  DropoutNet    +0,0103* (+12,4%) [+0,0064; +0,0141]; p_Holm = 5,0 × 10⁻¹⁹; 1.504 HV khác 0     Tăng có ý nghĩa                          —
       E-RQ1 (Họ chính: Gốc + LLM so với Gốc)                                        ALDI             −0,0018 (−2,2%) [−0,0050; +0,0012]; p_Holm = 1,00; 1.934 HV khác 0     Không khác biệt                          —
       E-RQ1 (Họ chính: Gốc + LLM so với Gốc)                                     SemProj    +0,0142* 

In [5]:
# Thống kê cơ chế của nhóm khóa trùng 
from src.models.coldrec_adapter import DUP_GROUPS_PARQUET

dup = pd.read_parquet(DUP_GROUPS_PARQUET)
grp = dict(zip(dup.course_id.astype(str).map(ad.item2idx), dup.dup_group_id))
S = {"raw": ad.S_raw, "llm": ad.S_llm}
n = hit = 0
cos = {"raw": [], "llm": []}
for u, items in ad.test_target_dict.items():
    seen = {}
    for j in ad.hist_trva.get(u, []):
        seen.setdefault(grp[j], j)
    for i in items:
        if i in ad.groups_prq3["trung"]:
            n += 1
            if grp[i] in seen:
                hit += 1
                for k in S:
                    cos[k].append(float(S[k][i] @ S[k][seen[grp[i]]]))
print(f"{hit}/{n} đích trùng ({hit / n:.1%}) là học lại khóa cùng nhóm trong lịch sử")
print({k: round(float(np.mean(v)), 3) for k, v in cos.items()})

300/1140 đích trùng (26.3%) là học lại khóa cùng nhóm trong lịch sử
{'raw': 0.732, 'llm': 0.628}
